In [1]:
# 01 - Lokasi proyek dan pengaturan notebook
# Notebook memakai kode, data, model, dan hasil yang ada dalam repositori.
# False meninjau hasil tersimpan. True menjalankan E0-E5 dan H3 dengan kode asli.
from pathlib import Path
import os, sys, json, hashlib, importlib, runpy
from unittest.mock import patch
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path.cwd(), *Path.cwd().parents, Path(r'D:\FILE AND TASK\TA')]
ROOT = next((p.resolve() for p in candidates if (p / 'data/manifests/dataset_split.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari folder repositori TA yang memiliki data/manifests/dataset_split.csv')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
JALANKAN_ULANG = False
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
print('Proyek:', ROOT)
print('Mode:', 'run baru' if JALANKAN_ULANG else 'membaca dan memeriksa hasil tersimpan')


Proyek: D:\FILE AND TASK\TA
Mode: membaca dan memeriksa hasil tersimpan


In [2]:
# 02 - Input: manifes target, derau, dan kontrol
split = pd.read_csv(ROOT / 'data/manifests/dataset_split.csv', dtype={'recording_id': str})
noise = pd.read_csv(ROOT / 'data/manifests/itera_noise_manifest.csv')
unknown = pd.read_csv(ROOT / 'data/manifests/unknown_open_set_manifest.csv', dtype={'unknown_id': str})
gallery = split.loc[split.split_role.eq('gallery')].reset_index(drop=True)
query = split.loc[split.split_role.eq('query_clean')].reset_index(drop=True)
calibration = split.loc[split.split_role.eq('calibration')].reset_index(drop=True)
unknown_cal = unknown.loc[unknown.split_role.eq('unknown_calibration')].reset_index(drop=True)
unknown_test = unknown.loc[unknown.split_role.eq('unknown_test')].reset_index(drop=True)
assert (len(split), split.species_key.nunique(), len(gallery), len(query), len(calibration)) == (1216, 20, 673, 311, 232)
assert (len(noise), len(unknown_cal), len(unknown_test)) == (1799, 233, 310)
display(pd.DataFrame({'kelompok':['target','galeri','kueri uji','kalibrasi target','derau Itera','kalibrasi kontrol','uji kontrol'], 'jumlah':[len(split),len(gallery),len(query),len(calibration),len(noise),len(unknown_cal),len(unknown_test)]}))
display(split.groupby(['species_key','scientific_name']).size().rename('jumlah_rekaman').reset_index())


,kelompok,jumlah
0,target,1216
1,galeri,673
2,kueri uji,311
3,kalibrasi target,232
4,derau Itera,1799
5,kalibrasi kontrol,233
6,uji kontrol,310


,species_key,scientific_name,jumlah_rekaman
0,ashtai1,Orthotomus ruficeps,30
1,colkin1,Todiramphus chloris,40
2,comior1,Aegithina tiphia,39
3,crseag1,Spilornis cheela,35
4,ghcfly1,Culicicapa ceylonensis,38
5,gpipig1,Ducula aenea,45
6,hacdro1,Dicrurus hottentottus,91
7,hoopit1,Pitta sordida,62
8,horbab1,Malacocincla sepiaria,68
9,latnig1,Caprimulgus macrurus,47


In [3]:
# 03 - Audit identitas dan ketersediaan input
assert split.recording_id.is_unique and unknown.unknown_id.is_unique
roles = {name: set(part.author.dropna().astype(str).str.strip()) for name, part in split.groupby('split_role')}
for first, second in [('gallery','query_clean'), ('gallery','calibration'), ('query_clean','calibration')]:
    assert not (roles[first] & roles[second]), (first, second)
missing_target = [p for p in split.file_path if not (ROOT / str(p)).is_file()]
missing_unknown = [p for p in unknown.file_path if not (ROOT / str(p)).is_file()]
missing_noise = [p for p in noise.filename if not (ROOT / 'data/itera_noise' / str(p)).is_file()]
models = [ROOT / 'checkpoints/Cnn14_mAP=0.431.pth', ROOT / 'checkpoints/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx']
display(pd.DataFrame({'input':['audio target','kontrol','bank derau','checkpoint'], 'hilang':[len(missing_target),len(missing_unknown),len(missing_noise),sum(not p.is_file() for p in models)]}))
print('Pemisahan nama perekam diperiksa pada data target; ini bukan bukti bebas overlap pada seluruh sesi atau data pralatih.')


,input,hilang
0,audio target,1216
1,kontrol,543
2,bank derau,1799
3,checkpoint,2


Pemisahan nama perekam diperiksa pada data target; ini bukan bukti bebas overlap pada seluruh sesi atau data pralatih.


In [4]:
# 04 - E0: prapemrosesan dan audit 100 rekaman
# Fungsi asli memuat mono 32 kHz, memilih jendela energi, padding, normalisasi RMS, dan pembatasan amplitudo.
E0 = ROOT / 'results/reproductions/E0_20261011_202557'
if JALANKAN_ULANG:
    if missing_target or missing_noise or missing_unknown or any(not p.is_file() for p in models):
        raise FileNotFoundError('Lengkapi audio, derau, kontrol, dan checkpoint sebelum menjalankan ulang.')
    E0 = importlib.import_module('scripts.run_e0_preprocessing_verification').main()
display(pd.read_csv(E0 / 'tables/summary.csv'))
display(pd.read_csv(E0 / 'tables/representation_probe.csv'))
print('Kode prapemrosesan:', ROOT / 'src/preprocess.py')


,metric,value
0,audit_rows,100.000000
1,unique_species,20.000000
2,native_duration_min_seconds,4.477458
3,native_duration_median_seconds,31.864816
4,native_duration_max_seconds,199.152646
5,extension_container_mismatches,6.000000
6,soundfile_decode_errors,0.000000
7,preprocess_length_failures,0.000000
8,preprocess_nonfinite_failures,0.000000
9,target_rms,0.050000


,representation,expected_dimension,actual_dimension,finite,l2_norm
0,R0,40,40,True,1.0
1,R1,2048,2048,True,1.0
2,R2,1024,1024,True,1.0
3,R3,40,40,True,1.0


Kode prapemrosesan: D:\FILE AND TASK\TA\src\preprocess.py


In [5]:
# 05 - Menyiapkan representasi galeri dan kueri
# R0 MFCC 40 nilai; R1 PANNs 2.048; R2 BirdNET 1.024; R3 acak 40 berdasarkan ID.
# Model pralatih R1/R2 tidak diperbarui. R3 tidak mengekstrak ciri dari waveform.
# E1-E3 memakai representasi yang dibuat dari audio/ID pada run ini.
cache_dir = ROOT / 'results/features'
if JALANKAN_ULANG:
    cache_dir = ROOT / 'results/reproductions' / f'NOTEBOOK_CLEAN_{stamp}'
    importlib.import_module('scripts.rebuild_e4_clean_caches').rebuild(cache_dir, check_only=False)
cache = {}
for rep, dim in [('R0',40),('R1',2048),('R2',1024),('R3',40)]:
    path = cache_dir / f'clean_embeddings_{rep}.npz'
    if not path.is_file():
        raise FileNotFoundError(f'Cache {rep} tidak tersedia: {path}')
    with np.load(path, allow_pickle=False) as data:
        cache[rep] = {key: data[key] for key in data.files}
    assert cache[rep]['gal_feats'].shape == (673,dim)
    assert cache[rep]['qry_feats'].shape == (311,dim)
    assert np.array_equal(cache[rep]['gal_ids'].astype(str), gallery.recording_id.astype(str).to_numpy())
    assert np.array_equal(cache[rep]['qry_ids'].astype(str), query.recording_id.astype(str).to_numpy())
    print(rep, 'galeri',cache[rep]['gal_feats'].shape,'kueri',cache[rep]['qry_feats'].shape)


FileNotFoundError: Cache R0 tidak tersedia: D:\FILE AND TASK\TA\results\features\clean_embeddings_R0.npz

In [ ]:
# 06 - E1: pencarian pada kondisi tanpa penambahan derau
E2_ASLI = ROOT / 'results/reproductions/E2_controlled_20261012T021600_897497Z'
if JALANKAN_ULANG:
    runner = importlib.import_module('scripts.materialize_e1_from_cache')
    with patch.object(runner, 'CACHE_DIR', cache_dir):
        E1 = {rep: runner.materialize(rep) for rep in cache}
    e1_summary = [
        {'representation': rep, **json.loads((path / 'metadata.json').read_text(encoding='utf-8'))['summary']}
        for rep, path in E1.items()
    ]
    display(pd.DataFrame(e1_summary))
else:
    reference = pd.read_csv(E2_ASLI / 'tables/e2_snr_robustness.csv')
    display(reference.loc[reference.condition.eq('Clean'), ['representation','mAP@10','Precision@10','Recall@10','MRR','Top1_Accuracy']])


In [ ]:
# 07 - E2: pencampuran derau Itera pada empat SNR
E2 = E2_ASLI
if JALANKAN_ULANG:
    runner = importlib.import_module('scripts.run_e2_controlled_noise')
    E2 = ROOT / 'results/reproductions' / f'E2_controlled_{stamp}'
    with patch.object(runner, 'FEATURES', cache_dir):
        runner.execute(E2, resume=False)
e2 = pd.read_csv(E2 / 'tables/e2_snr_robustness.csv')
display(e2[['representation','condition','mAP@10','relative_retention','Top1_Accuracy','Recall@10','Precision@10','MRR']])


In [ ]:
# 08 - Pemeriksaan ulang mAP@10 E2 dari skor setiap kueri
e2_check = []
for rep in ['R0','R1','R2','R3']:
    for condition in ['Clean','SNR_20dB','SNR_10dB','SNR_0dB','SNR_-5dB']:
        raw = pd.read_csv(E2 / 'raw' / f'{rep}_{condition}.csv')
        assert len(raw) == 311 and raw.query_id.astype(str).nunique() == 311
        expected = float(e2.loc[e2.representation.eq(rep) & e2.condition.eq(condition),'mAP@10'].iloc[0])
        calculated = float(raw['AP@10'].mean())
        assert np.isclose(expected, calculated, atol=1e-10)
        e2_check.append({'representasi':rep,'kondisi':condition,'mAP@10':calculated})
display(pd.DataFrame(e2_check))


In [ ]:
# 09 - Grafik E2: perubahan kualitas pencarian menurut SNR
fig, ax = plt.subplots(figsize=(8,4.5))
for rep, group in e2.loc[e2.condition.ne('Clean')].groupby('representation'):
    group = group.sort_values('snr_db', ascending=False)
    ax.plot(group.snr_db, group['mAP@10'], marker='o', label=rep)
ax.set(xlabel='SNR (dB)', ylabel='mAP@10', title='Kualitas pencarian pada kondisi berderau')
ax.invert_xaxis(); ax.grid(alpha=.3); ax.legend(); plt.show()


In [ ]:
# 10 - E3: kalibrasi ambang dan pengujian penerimaan
# Ambang dari kalibrasi tanpa penambahan derau dipertahankan untuk semua kondisi uji.
# Data target pada E3 menggunakan hasil E2 dari run yang sama.
E3 = ROOT / 'results/reproductions/E3_open_set_20261012T022548_802536Z'
if JALANKAN_ULANG:
    runner = importlib.import_module('scripts.run_e3_open_set')
    with patch.object(runner, 'FEATURES', cache_dir):
        E3 = runner.run(E2)
thresholds = pd.read_csv(E3 / 'tables/frozen_thresholds.csv')
e3 = pd.read_csv(E3 / 'tables/threshold_transfer.csv')
display(thresholds)
display(e3[['representation','condition','frozen_tau','Recall_at_tau','False_Positive_Rate']])


In [ ]:
# 11 - Pemeriksaan keputusan E3 dari skor individu
e3_scores = pd.read_csv(E3 / 'raw/test_scores.csv')
assert ((e3_scores.similarity_score >= e3_scores.frozen_tau).astype(int) == e3_scores.predicted_accept).all()
checks = e3_scores.groupby(['representation','condition','target_type']).predicted_accept.agg(['mean','size']).reset_index()
display(checks.rename(columns={'mean':'proporsi_diterima','size':'jumlah_kueri'}))
print('Penerimaan target pada E3 tidak otomatis berarti spesies Top-1 benar.')


In [ ]:
# 12 - E4: uji lima lokasi pada SNR yang sama
# Runner asli memeriksa cache acuan, checkpoint dan identitas derau sebelum inferensi.
E4 = ROOT / 'results/reproductions/E4_5sites_20261011T213404_350219Z'
if JALANKAN_ULANG:
    script = ROOT / 'scripts/run_e4_five_locations_v10_stage7.py.draft'
    with patch.object(sys, 'argv', [str(script)]):
        E4 = runpy.run_path(str(script), run_name='__main__')['E4_RUN_DIR']
e4_frames = []
for rep in ['R0','R1','R2','R3']:
    df = pd.read_csv(E4 / 'raw' / f'E4_{rep}_raw.csv', low_memory=False)
    assert len(df.loc[df.condition.eq('Clean')]) == 311
    noisy = df.loc[df.condition.ne('Clean')].copy()
    assert len(noisy) == 311 * 5 * 4 * 4
    assert noisy.habitat.nunique() == 5 and noisy.repeat_id.nunique() == 4
    e4_frames.append(noisy)
e4 = pd.concat(e4_frames, ignore_index=True)
e4_summary = e4.groupby(['representation','habitat','snr_db'],as_index=False)['AP@10'].mean().rename(columns={'AP@10':'mAP@10'})
display(e4_summary.head(20))
print('Baris pengujian E4:',len(e4),'; pengulangan bukan kueri independen baru.')


In [ ]:
# 13 - Analisis sensitivitas H3 dan bootstrap kueri berpasangan
# H3 dibandingkan pada skala absolut dan relatif; subset tanpa Kebun Raya bersifat eksploratif.
H3 = ROOT / 'results/reproductions/H3_sensitivity_20261015'
if JALANKAN_ULANG:
    H3 = ROOT / 'results/reproductions' / f'H3_notebook_{stamp}'
    runner = importlib.import_module('scripts.run_h3_sensitivity_reanalysis')
    with patch.object(runner, 'E4', E4):
        runner.run(H3, iterations=10000)
h3 = pd.read_csv(H3 / 'tables/h3_swing_summary.csv')
display(h3)
print('Bootstrap mengambil ulang kueri; lokasi dan segmen derau tetap, sehingga tidak mengukur ketidakpastian lokasi baru.')


In [ ]:
# 14 - E5: diagnostik kegagalan berdasarkan E2, E3, dan data E4 yang tersedia
E5 = ROOT / 'results/reproductions/E5_diagnostics_20261012T091512_868500Z'
if JALANKAN_ULANG:
    E5 = importlib.import_module('scripts.run_e5_diagnostics').main(E2, E3, E4)
display(pd.read_csv(E5 / 'tables/failure_mode_summary.csv').head(30))
display(pd.read_csv(E5 / 'tables/selected_case_manifest.csv').head(10))
print('Fitur akustik hasil rekonstruksi tersedia untuk E2; kolom yang tidak tersedia pada E4 tidak direka.')


In [ ]:
# 15 - Audit pendengaran manusia pada 120 kasus E2 pilihan
# Di bawah ini adalah ulasan manusia 30 Oktober atas kasus dari run E5 sebelumnya.
audit_path = ROOT / 'Rencana-eksperimen-bimbingan/audit-pendengaran-E5/E5_HASIL_ULASAN_2026-10-30.csv'
audit = pd.read_csv(audit_path)
assert len(audit) == 120 and audit.case_id.nunique() == 120
assert audit.review_status.eq('reviewed').all()
display(audit.groupby(['failure_mode_computational','query_vocalisation_audible']).size().rename('jumlah').reset_index())
display(audit[['query_vocalisation_audible','top1_vocalisation_audible']].apply(pd.Series.value_counts))
print('Audit 30 Oktober: 120 kasus dari run E5 asal dinilai secara deskriptif oleh satu pendengar.')


In [ ]:
# 16 - Jejak berkas dan status penelitian
for name, path in [('E0',E0),('E2',E2),('E3',E3),('E4',E4),('H3',H3),('E5',E5)]:
    meta = path / 'metadata.json'
    data = json.loads(meta.read_text(encoding='utf-8')) if meta.is_file() else {}
    print(name, '|', data.get('status', data.get('analysis','lihat metadata')), '|', path)

print('Belum diuji: burung target direkam langsung dalam soundscape Itera dan pengaruh pita frekuensi yang diisolasi.')
print('Batas: satu pendengar, ketidakpastian pemisahan sesi unknown, kemungkinan overlap prapelatihan, dan generalisasi lokasi baru.')
print('H3/H4 yang dirumuskan setelah hasil tersedia dilaporkan sebagai eksploratif.')
